# Outline

- Test the best model from model selection
- Make Pickle file for pipeline and model

In [1]:
import numpy as np
import pandas as pd


In [57]:
X_train = pd.read_csv('../Missing Values Imputation/Houses/X_train.csv',index_col=0)
X_test = pd.read_csv('../Missing Values Imputation/Houses/X_test.csv',index_col=0)
y_train = pd.read_csv('../Missing Values Imputation/Houses/y_train.csv',index_col=0)
y_test = pd.read_csv('../Missing Values Imputation/Houses/y_test.csv',index_col=0)

In [58]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor

In [59]:
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score,mean_absolute_error
from category_encoders import TargetEncoder
from sklearn.base import BaseEstimator, TransformerMixin

In [60]:
from sklearn import set_config

set_config(transform_output='pandas')

In [70]:


class CustomOrdinalMapperHouses(BaseEstimator, TransformerMixin):
    def __init__(self, mappings):
        self.mappings = mappings
        
    def fit(self, X, y=None):
        return self
        
    def transform(self, X):
        # Handle case if X is a NumPy array, convert to DataFrame for column mapping
        if not isinstance(X, pd.DataFrame):
            X_copy = pd.DataFrame(X)
        else:
            X_copy = X.copy()
            
        for col, mapping_dict in self.mappings.items():
            if col in X_copy.columns:
                X_copy[col] = X_copy[col].map(mapping_dict).fillna(1).astype(int)
        return X_copy

    def get_feature_names_out(self, input_features=None):
        """Returns the input features as the output features."""
        if input_features is None:
            return np.array(list(self.mappings.keys()))
        return np.asarray(input_features)


class LogTransformer(BaseEstimator, TransformerMixin):
    def __init__(self):
        """Custom Transformer for Log Transformation."""
        pass

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_copy = np.copy(X)
        return np.log1p(X_copy)
        
    def inverse_transform(self, X):
        X_copy = np.copy(X)
        return np.expm1(X_copy) 

    def get_feature_names_out(self, input_features=None):
        """Returns the input features as the output features."""
        if input_features is None:
            raise ValueError("input_features must be provided to get_feature_names_out.")
        return np.asarray(input_features)

all_mappings_houses = {
    'Property era': {
        'Vintage': 1, 'Established': 2, 'Recently Built': 3, 
        'Modern Era': 4, 'Brand New (2025)': 5, 'Future': 6
    }
}

In [71]:
categorical_cols = ['Property era']

In [72]:
prepocessor = ColumnTransformer(
    [
        ('Custom Ordinal Encoder',CustomOrdinalMapper(all_mappings),categorical_cols),
        ('Target Encoder',TargetEncoder(smoothing=20,min_samples_leaf=2,handle_unknown='value'),['Main Location']),
        ('Log Transform',LogTransformer(),['Area(Marla)'])
    ],
    remainder='passthrough',verbose_feature_names_out=False
)

In [73]:
prepocessor.fit_transform(X_train,y_train)

,Property era,Main Location,Area(Marla),Bath(s),Bedroom(s),Servant Quarters,Kitchens,Store Rooms,Storey Unit,IsPrimeLoc,SolarInstalled,WaterBore,CornerHouse,luxury_type
906,5,11.632752,2.079442,5.0,5.0,1.0,2.0,1.0,2.0,False,False,False,False,0
3379,5,4.006512,2.397895,6.0,5.0,1.0,2.0,1.0,2.0,False,False,False,False,0
2094,5,3.792289,1.791759,4.0,3.0,1.0,2.0,1.0,2.0,True,False,False,True,0
218,5,9.301197,2.708050,6.0,6.0,1.0,1.0,1.0,2.0,True,False,False,False,0
3455,5,7.172146,1.609438,5.0,5.0,1.0,2.0,1.0,2.0,True,False,False,False,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5175,3,11.449320,3.044522,5.0,5.0,2.0,2.0,2.0,2.0,False,False,False,False,0
686,5,7.172146,3.044522,7.0,6.0,1.0,1.0,1.0,2.0,False,False,False,False,1
4631,5,7.172146,1.609438,4.0,4.0,1.0,2.0,1.0,2.0,False,False,False,False,0
5701,3,40.374595,3.295837,7.0,6.0,2.0,2.0,2.0,2.0,False,False,False,False,0


In [74]:
y_train_log = np.log1p(y_train)

In [75]:
X_train_transformed = prepocessor.fit_transform(X_train,y_train_log['Price(Cr)'])
X_test_transformed = prepocessor.transform(X_test)

In [76]:
overfitted_params = {'n_estimators': 1000,
 'max_depth': 15,
 'learning_rate': 0.00505825958265549,
 'subsample': 0.8944998368111671,
 'colsample_bytree': 0.9305368372411382,
 'colsample_bylevel': 0.7407153424980754,
 'gamma': 0.021373237297058784,
 'min_child_weight': 4,
 'reg_alpha': 0.005116450915274579,
 'reg_lambda': 0.0007460273123214234,
 'grow_policy': 'lossguide',
 'max_leaves': 52}

In [77]:
best_model = XGBRegressor(**overfitted_params)
best_model.fit(X_train_transformed,y_train_log['Price(Cr)'])
y_pred = best_model.predict(X_test_transformed)
y_pred = np.expm1(y_pred)
r2_score(y_test,y_pred)

0.9317037463188171

In [78]:
mean_absolute_error(y_test,y_pred)

1.3294177055358887

In [21]:
test_input = X_train.iloc[:5]

In [22]:
test_input_transformed = prepocessor.transform(test_input)

In [23]:
test_input_transformed

,Property era,Main Location,Area(Marla),Bath(s),Bedroom(s),Servant Quarters,Kitchens,Store Rooms,Storey Unit,IsPrimeLoc,SolarInstalled,WaterBore,CornerHouse,luxury_type
906,5,2.383903,2.079442,5.0,5.0,1.0,2.0,1.0,2.0,False,False,False,False,0
3379,5,1.555632,2.397895,6.0,5.0,1.0,2.0,1.0,2.0,False,False,False,False,0
2094,5,1.497756,1.791759,4.0,3.0,1.0,2.0,1.0,2.0,True,False,False,True,0
218,5,2.243517,2.708050,6.0,6.0,1.0,1.0,1.0,2.0,True,False,False,False,0
3455,5,2.012407,1.609438,5.0,5.0,1.0,2.0,1.0,2.0,True,False,False,False,0


In [24]:
y_pred_test = best_model.predict(test_input_transformed)
y_pred_test = np.expm1(y_pred_test)
r2_score(y_train[:5],y_pred_test)

0.7255208492279053

In [25]:
mean_absolute_error(y_train[:5],y_pred_test)

1.1759672164916992

In [26]:
y_pred_test

array([ 7.6353188,  4.8600106,  2.309601 , 11.413257 ,  4.051489 ],
      dtype=float32)

In [27]:
y_train[:5]

,Price(Cr)
906,9.75
3379,4.50
2094,2.60
218,8.85
3455,3.50


In [79]:
model_pipeline = Pipeline(
    [
        ('Preprocessor',prepocessor),
        ('Model',best_model)
    ]
)

In [80]:
model_pipeline.fit(X_train,y_train_log['Price(Cr)'])

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('Preprocessor', ...), ('Model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('Custom Ordinal Encoder', ...), ('Target Encoder', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If t

In [81]:
np.expm1(model_pipeline.predict(X_train[:5]))

array([ 7.6353188,  4.8600106,  2.309601 , 11.413257 ,  4.051489 ],
      dtype=float32)

In [82]:
# Dump the best pipeline
import joblib

joblib.dump(model_pipeline,'pipeline_pickle_houses.joblib')

['pipeline_pickle_houses.joblib']

In [34]:
model = joblib.load('pipeline_pickle_houses.joblib')

In [35]:
y_pred =model.predict(X_test)
y_pred = np.expm1(y_pred)
r2_score(y_test,y_pred)

0.9317037463188171

In [36]:
mean_absolute_error(y_test,y_pred)

1.3294177055358887

In [38]:
X_train_transformed.columns

Index(['Property era', 'Main Location', 'Area(Marla)', 'Bath(s)', 'Bedroom(s)',
       'Servant Quarters', 'Kitchens', 'Store Rooms', 'Storey Unit',
       'IsPrimeLoc', 'SolarInstalled', 'WaterBore', 'CornerHouse',
       'luxury_type'],
      dtype='str')

In [42]:
X_train_transformed['Kitchens'].unique()

array([2., 1., 3.])

In [41]:
X_test_transformed['Kitchens'].unique()

array([1., 2., 3.])

In [85]:
X_train['IsPrimeLoc'].value_counts().index

Index([False, True], dtype='bool', name='IsPrimeLoc')